In [59]:
import polars as pl
import numpy as np
import math
import plotly.express as px
import plotly.graph_objects as go

In [60]:
df = pl.read_excel("./data/task2.xlsx", sheet_name="В-18")

df.head()

Х1,Х2,Х3,Х4,Х5,Х6,Х7,Х8,Х9,Y
i64,f64,f64,f64,f64,i64,f64,f64,f64,f64
63,82.787941,479.22016,234.368287,314.954529,41,173.541475,228.767113,284.431898,7098.74738
31,108.811008,754.38713,158.621085,261.953308,178,236.837209,159.587744,300.041229,13014.307845
36,90.300346,478.592271,90.131814,236.620393,38,226.652648,231.73432,295.117184,8748.641969
36,91.904692,719.830328,217.716565,412.566963,158,228.631988,172.016608,296.762484,11394.990417
72,90.251698,657.428365,193.847045,332.99756,128,157.035283,186.7129,281.550313,10154.04808


In [61]:
df.describe()

statistic,Х1,Х2,Х3,Х4,Х5,Х6,Х7,Х8,Х9,Y
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",400.0,400.0,400.0,400.0,400.0,400.0,400.0,400.0,400.0,400.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",50.97,99.801038,657.024349,152.00945,298.290133,128.9,197.899886,185.603806,289.439978,10920.12504
"""std""",27.444162,15.808473,149.796361,41.42786,82.596431,74.844433,54.779154,37.569878,13.765893,2647.88559
"""min""",1.0,49.570253,402.489561,47.649812,92.325761,1.0,100.732909,124.557276,262.450023,5131.805871
"""25%""",29.0,89.10881,521.221214,123.092325,242.472639,61.0,153.076708,151.778647,278.188869,8649.632451
"""50%""",51.0,99.291521,660.458752,151.722231,300.252268,133.0,198.759303,183.940731,288.980345,11076.44901
"""75%""",74.0,110.081673,789.491008,181.551927,352.446694,195.0,240.895284,219.660763,300.406114,13014.307845
"""max""",99.0,148.304191,897.209732,295.972554,522.329574,249.0,298.124643,250.877451,316.563645,16017.916121


In [62]:
def get_col_stats(col: pl.Series) -> dict:
    n = col.len()
    min = col.min()
    max = col.max()
    r = max - min
    k = 1 + math.floor(3.222 * np.log10(n)) # правило Стёрджеса с округлением вниз
    h = r / k

    result = {
        "n": n,
        "min": min,
        "max": max,
        "r": r,
        "k": k,
        "h": h
    }

    return result

In [63]:
def get_bins(col_stats: dict) -> np.ndarray:
    
    min_val = col_stats["min"]
    h = col_stats["h"]
    k = col_stats["k"]
    max_val = col_stats["max"]

    bins = min_val + np.arange(k) * h
    bins = np.append(bins, max_val)
    
    return bins

In [64]:
def get_interval_table(col: pl.Series) -> pl.DataFrame:
    stats = get_col_stats(col)
    bins = get_bins(stats)

    groups = np.digitize(col.to_numpy(), bins[1:-1], right=False)

    interval_left = bins[:-1]
    interval_right = bins[1:]

    df = pl.DataFrame({
        "value": col,
        "group": groups
    })

    group_stats = (
        df.group_by("group")
        .agg(
            pl.len().alias("freq"),
            pl.col("value").mean().alias("mean")
        )
        .with_columns(
            (pl.col("freq") / len(df)).alias("w")
        )
        .sort("group")
    )

    group_stats = group_stats.with_columns(
        pl.col("freq").cum_sum().alias("cum_freq"),
        pl.col("w").cum_sum().alias("cum_w")
    )

    result = pl.DataFrame({
        "group": list(range(stats["k"])),
        "interval_left": interval_left,
        "interval_right": interval_right
    })

    result = result.join(group_stats, on="group", how="left")

    result = result.select(["interval_left", "interval_right", "freq", "w", "cum_freq", "cum_w", "mean"])
   
    return result

In [65]:
def plot_freq_bar(interval_table: pl.DataFrame, title: str) -> None:
    fig = px.bar(
        interval_table,
        x="interval_right",
        y="freq",
        title=f"Гистограмма частот для {title}",
        labels={
            "interval_right": "Интервалы",
            "freq": "Частота"
        },
        width=800,
        height=400,
        text_auto=True
    )

    fig.update_layout(
        bargap=0
    )

    tick_labels = [
        f"[{l:.3g}, {r:.3g})" if i < len(interval_table)-1 else f"[{l:.3g}, {r:.3g}]"
        for i, (l, r) in enumerate(zip(
            interval_table["interval_left"],
            interval_table["interval_right"]
        ))
    ]

    fig.update_layout(
        xaxis=dict(
            tickmode="array",
            tickvals=interval_table.get_column("interval_right"),
            ticktext=tick_labels,
            tickangle=45
        )
    )
    
    fig.show()

In [66]:
def plot_polygon(interval_table: pl.DataFrame, title: str) -> None:
    fig = px.line(
        interval_table,
        x="interval_right", 
        y="cum_w", 
        title=f"Полигон накопленных частотностей для {title}", 
        labels={
            "interval_right": "Правая граница интервала", 
            "cum_w": "Накопленная относительная частота"
        },
        width=800,
        height=400,
        markers=True
    )
    
    fig.show()

In [67]:
def get_origin_binned_comparison(origin_col: pl.Series, binned: pl.DataFrame) -> pl.DataFrame:
    mean_origin = origin_col.mean()
    mean_binned = binned.select(
        pl.col("w").mul(pl.col("mean")).sum()
    ).item()

    var_origin = origin_col.var(ddof=0)
    var_binned = binned.select(
        (pl.col("mean").sub(mean_binned)).pow(2).mul(pl.col("w")).sum()
    ).item()

    std_origin = np.sqrt(var_origin)
    std_binned = np.sqrt(var_binned)

    skewness_origin = origin_col.skew()
    skewness_binned = binned.select(
        (pl.col("mean").sub(mean_binned)).pow(3).mul(pl.col("w")).sum().truediv(std_binned ** 3)
    ).item()

    excess_origin = origin_col.kurtosis()
    excess_binned = binned.select(
        (pl.col("mean").sub(mean_binned)).pow(4).mul(pl.col("w")).sum().truediv(std_binned ** 4).sub(3)
    ).item()

    result = {
        "source": ["origin", "binned"],
        "mean": [mean_origin, mean_binned],
        "variance": [var_origin, var_binned],
        "std": [std_origin, std_binned],
        "skewness": [skewness_origin, skewness_binned],
        "excess": [excess_origin, excess_binned]
    }

    result_df = pl.DataFrame(result)

    return result_df

In [68]:
def get_estimation_of_population(binned: pl.DataFrame, stats: dict) -> pl.DataFrame:
    mean = binned.select(
        pl.col("w").mul(pl.col("mean")).sum()
    ).item()

    temp = binned.select(
        (pl.col("mean").sub(mean)).pow(2)
    ).to_series()

    var = binned.select(
        pl.col("w").mul(temp).sum()
    ).item()

    std = np.sqrt(var)

    n = stats["n"] - 1
    std_unbiased = binned.select(
        pl.col("freq").truediv(n).mul(temp).sum()
    ).item()

    std_unbiased = np.sqrt(std_unbiased)

    result = {
        "mean": mean,
        "std": std,
        "std_unbiased": std_unbiased
    }

    result_df = pl.DataFrame(result)

    return result_df

# Задание 1

## Х1

In [69]:
x1 = df.get_column("Х1")

In [70]:
x1_stats = get_col_stats(x1)
x1_stats

{'n': 400, 'min': 1, 'max': 99, 'r': 98, 'k': 9, 'h': 10.88888888888889}

In [71]:
x1_bins = get_bins(x1_stats)
x1_bins

array([ 1.        , 11.88888889, 22.77777778, 33.66666667, 44.55555556,
       55.44444444, 66.33333333, 77.22222222, 88.11111111, 99.        ])

In [72]:
x1_intervals = get_interval_table(x1)
x1_intervals

interval_left,interval_right,freq,w,cum_freq,cum_w,mean
f64,f64,u32,f64,u32,f64,f64
1.0,11.888889,39,0.0975,39,0.0975,6.358974
11.888889,22.777778,34,0.085,73,0.1825,17.117647
22.777778,33.666667,47,0.1175,120,0.3,27.93617
33.666667,44.555556,50,0.125,170,0.425,38.18
44.555556,55.444444,47,0.1175,217,0.5425,49.404255
55.444444,66.333333,55,0.1375,272,0.68,61.854545
66.333333,77.222222,36,0.09,308,0.77,71.138889
77.222222,88.111111,52,0.13,360,0.9,82.788462
88.111111,99.0,40,0.1,400,1.0,93.65


In [73]:
plot_freq_bar(x1_intervals, "Х1")

In [74]:
plot_polygon(x1_intervals, "Х1")

In [75]:
get_origin_binned_comparison(x1, x1_intervals)

source,mean,variance,std,skewness,excess
str,f64,f64,f64,f64,f64
"""origin""",50.97,751.2991,27.409836,-0.036755,-1.116182
"""binned""",50.97,741.196784,27.224929,-0.038344,-1.150434


Предположительно, данное распределение подчиняется равномерному закону, так как все значения колеблются около среднего значения, эксцесс близок к теоретическому эксцессу непрерывного распределения (-1.2), и при этом почти нулевая асимметрия, то есть распределение симметричное, что исключает скошенные законы.

In [76]:
get_estimation_of_population(x1_intervals, x1_stats)

mean,std,std_unbiased
f64,f64,f64
50.97,27.224929,27.259025


## Х2

In [77]:
x2 = df.get_column("Х2")

In [78]:
x2_stats = get_col_stats(x2)
x2_stats

{'n': 400,
 'min': 49.570253434308825,
 'max': 148.3041907243487,
 'r': 98.73393729003988,
 'k': 9,
 'h': 10.970437476671098}

In [79]:
x2_bins = get_bins(x2_stats)
x2_bins

array([ 49.57025343,  60.54069091,  71.51112839,  82.48156586,
        93.45200334, 104.42244082, 115.39287829, 126.36331577,
       137.33375325, 148.30419072])

In [80]:
x2_intervals = get_interval_table(x2)
x2_intervals

interval_left,interval_right,freq,w,cum_freq,cum_w,mean
f64,f64,u32,f64,u32,f64,f64
49.570253,60.540691,3,0.0075,3,0.0075,54.007102
60.540691,71.511128,10,0.025,13,0.0325,67.594106
71.511128,82.481566,39,0.0975,52,0.13,77.979135
82.481566,93.452003,87,0.2175,139,0.3475,88.286498
93.452003,104.422441,111,0.2775,250,0.625,98.905225
104.422441,115.392878,82,0.205,332,0.83,109.506214
115.392878,126.363316,48,0.12,380,0.95,119.87237
126.363316,137.333753,15,0.0375,395,0.9875,129.309917
137.333753,148.304191,5,0.0125,400,1.0,141.765829


In [81]:
plot_freq_bar(x2_intervals, "Х2")

In [82]:
plot_polygon(x2_intervals, "Х2")

In [83]:
get_origin_binned_comparison(x2, x2_intervals)

source,mean,variance,std,skewness,excess
str,f64,f64,f64,f64,f64
"""origin""",99.801038,249.283044,15.7887,0.036036,0.033651
"""binned""",99.801038,239.468311,15.474764,0.038912,0.039659


Предположительно, данное распределение подчиняется нормальному закону распределения, коэффициент асимметрии показывает незначительную правостороннюю асимметрию, эксцесс близок к нулю.

In [84]:
get_estimation_of_population(x2_intervals, x2_stats)

mean,std,std_unbiased
f64,f64,f64
99.801038,15.474764,15.494143


## Х3

In [85]:
x3 = df.get_column("Х3")

In [86]:
x3_stats = get_col_stats(x3)
x3_stats

{'n': 400,
 'min': 402.4895611409008,
 'max': 897.2097317980478,
 'r': 494.720170657147,
 'k': 9,
 'h': 54.96890785079411}

In [87]:
x3_bins = get_bins(x3_stats)
x3_bins

array([402.48956114, 457.45846899, 512.42737684, 567.39628469,
       622.36519254, 677.33410039, 732.30300825, 787.2719161 ,
       842.24082395, 897.2097318 ])

In [88]:
x3_intervals = get_interval_table(x3)
x3_intervals

interval_left,interval_right,freq,w,cum_freq,cum_w,mean
f64,f64,u32,f64,u32,f64,f64
402.489561,457.458469,47,0.1175,47,0.1175,425.403613
457.458469,512.427377,47,0.1175,94,0.235,486.135449
512.427377,567.396285,38,0.095,132,0.33,541.196513
567.396285,622.365193,44,0.11,176,0.44,597.602064
622.365193,677.3341,33,0.0825,209,0.5225,651.953331
677.3341,732.303008,46,0.115,255,0.6375,711.19637
732.303008,787.271916,43,0.1075,298,0.745,758.839903
787.271916,842.240824,49,0.1225,347,0.8675,815.143249
842.240824,897.209732,53,0.1325,400,1.0,873.694668


In [89]:
plot_freq_bar(x3_intervals, "Х3")

In [90]:
plot_polygon(x3_intervals, "Х3")

In [91]:
get_origin_binned_comparison(x3, x3_intervals)

source,mean,variance,std,skewness,excess
str,f64,f64,f64,f64,f64
"""origin""",657.024349,22382.852416,149.608998,-0.074683,-1.258852
"""binned""",657.024349,22134.986155,148.778312,-0.076939,-1.289665


Аналогично Х1 данное распределение предположительно является равномерным.

In [92]:
get_estimation_of_population(x3_intervals, x3_stats)

mean,std,std_unbiased
f64,f64,f64
657.024349,148.778312,148.964634


## Х4

In [93]:
x4 = df.get_column("Х4")

In [94]:
x4_stats = get_col_stats(x4)
x4_stats

{'n': 400,
 'min': 47.649811963933274,
 'max': 295.9725535872362,
 'r': 248.3227416233029,
 'k': 9,
 'h': 27.591415735922546}

In [95]:
x4_bins = get_bins(x4_stats)
x4_bins

array([ 47.64981196,  75.2412277 , 102.83264344, 130.42405917,
       158.01547491, 185.60689064, 213.19830638, 240.78972212,
       268.38113785, 295.97255359])

In [96]:
x4_intervals = get_interval_table(x4)
x4_intervals

interval_left,interval_right,freq,w,cum_freq,cum_w,mean
f64,f64,u32,f64,u32,f64,f64
47.649812,75.241228,9,0.0225,9,0.0225,64.343606
75.241228,102.832643,42,0.105,51,0.1275,90.103182
102.832643,130.424059,74,0.185,125,0.3125,117.594451
130.424059,158.015475,96,0.24,221,0.5525,144.368392
158.015475,185.606891,88,0.22,309,0.7725,170.252418
185.606891,213.198306,63,0.1575,372,0.93,197.419998
213.198306,240.789722,24,0.06,396,0.99,225.994244
240.789722,268.381138,3,0.0075,399,0.9975,246.49731
268.381138,295.972554,1,0.0025,400,1.0,295.972554


In [97]:
plot_freq_bar(x4_intervals, "Х4")

In [98]:
plot_polygon(x4_intervals, "Х4")

In [99]:
get_origin_binned_comparison(x4, x4_intervals)

source,mean,variance,std,skewness,excess
str,f64,f64,f64,f64,f64
"""origin""",152.00945,1711.976887,41.376042,0.089822,-0.30148
"""binned""",152.00945,1653.643869,40.66502,0.095929,-0.332304


Аналогично Х2 данное распределение предположительно подчиняется нормальному распределению, но с чуть более ярко выраженной правосторонней асимметрией.

In [100]:
get_estimation_of_population(x4_intervals, x4_stats)

mean,std,std_unbiased
f64,f64,f64
152.00945,40.66502,40.715947


## Х5

In [101]:
x5 = df.get_column("Х5")

In [102]:
x5_stats = get_col_stats(x5)
x5_stats

{'n': 400,
 'min': 92.32576082557753,
 'max': 522.3295744527042,
 'r': 430.00381362712665,
 'k': 9,
 'h': 47.778201514125186}

In [103]:
x5_bins = get_bins(x5_stats)
x5_bins

array([ 92.32576083, 140.10396234, 187.88216385, 235.66036537,
       283.43856688, 331.2167684 , 378.99496991, 426.77317142,
       474.55137294, 522.32957445])

In [104]:
x5_intervals = get_interval_table(x5)
x5_intervals

interval_left,interval_right,freq,w,cum_freq,cum_w,mean
f64,f64,u32,f64,u32,f64,f64
92.325761,140.103962,13,0.0325,13,0.0325,124.49091
140.103962,187.882164,25,0.0625,38,0.095,163.387006
187.882164,235.660365,50,0.125,88,0.22,215.115644
235.660365,283.438567,86,0.215,174,0.435,259.141554
283.438567,331.216768,85,0.2125,259,0.6475,308.405455
331.216768,378.99497,72,0.18,331,0.8275,350.711116
378.99497,426.773171,42,0.105,373,0.9325,399.984127
426.773171,474.551373,22,0.055,395,0.9875,445.800931
474.551373,522.329574,5,0.0125,400,1.0,499.6845


In [105]:
plot_freq_bar(x5_intervals, "Х5")

In [107]:
plot_polygon(x5_intervals, "Х5")

In [108]:
get_origin_binned_comparison(x5, x5_intervals)

source,mean,variance,std,skewness,excess
str,f64,f64,f64,f64,f64
"""origin""",298.290133,6805.114976,82.493121,0.013826,-0.339221
"""binned""",298.290133,6619.405677,81.35973,0.011758,-0.351765


In [109]:
get_estimation_of_population(x5_intervals, x5_stats)

mean,std,std_unbiased
f64,f64,f64
298.290133,81.35973,81.461621


## Х6

In [110]:
x6 = df.get_column("Х6")

In [111]:
x6_stats = get_col_stats(x6)
x6_stats

{'n': 400, 'min': 1, 'max': 249, 'r': 248, 'k': 9, 'h': 27.555555555555557}

In [112]:
x6_bins = get_bins(x6_stats)
x6_bins

array([  1.        ,  28.55555556,  56.11111111,  83.66666667,
       111.22222222, 138.77777778, 166.33333333, 193.88888889,
       221.44444444, 249.        ])

In [113]:
x6_intervals = get_interval_table(x6)
x6_intervals

interval_left,interval_right,freq,w,cum_freq,cum_w,mean
f64,f64,u32,f64,u32,f64,f64
1.0,28.555556,46,0.115,46,0.115,12.5
28.555556,56.111111,46,0.115,92,0.23,42.543478
56.111111,83.666667,39,0.0975,131,0.3275,70.153846
83.666667,111.222222,43,0.1075,174,0.435,98.511628
111.222222,138.777778,33,0.0825,207,0.5175,125.606061
138.777778,166.333333,47,0.1175,254,0.635,154.702128
166.333333,193.888889,43,0.1075,297,0.7425,178.976744
193.888889,221.444444,50,0.125,347,0.8675,207.4
221.444444,249.0,53,0.1325,400,1.0,237.245283


In [114]:
plot_freq_bar(x6_intervals, "Х6")

In [115]:
plot_polygon(x6_intervals, "Х6")

In [116]:
get_origin_binned_comparison(x6, x6_intervals)

source,mean,variance,std,skewness,excess
str,f64,f64,f64,f64,f64
"""origin""",128.9,5587.685,74.750819,-0.08014,-1.25075
"""binned""",128.9,5525.846923,74.336041,-0.082066,-1.28091


In [128]:
get_estimation_of_population(x6_intervals, x6_stats)

mean,std,std_unbiased
f64,f64,f64
128.9,74.336041,74.429135


## Х7

In [119]:
x7 = df.get_column("Х7")

In [122]:
x7_stats = get_col_stats(x7)
x7_stats

{'n': 400,
 'min': 100.7329091936864,
 'max': 298.1246428341408,
 'r': 197.3917336404544,
 'k': 9,
 'h': 21.932414848939377}

In [123]:
x7_bins = get_bins(x7_stats)
x7_bins

array([100.73290919, 122.66532404, 144.59773889, 166.53015374,
       188.46256859, 210.39498344, 232.32739829, 254.25981314,
       276.19222799, 298.12464283])

In [124]:
x7_intervals = get_interval_table(x7)
x7_intervals

interval_left,interval_right,freq,w,cum_freq,cum_w,mean
f64,f64,u32,f64,u32,f64,f64
100.732909,122.665324,39,0.0975,39,0.0975,112.302377
122.665324,144.597739,52,0.13,91,0.2275,133.936606
144.597739,166.530154,36,0.09,127,0.3175,157.185257
166.530154,188.462569,56,0.14,183,0.4575,176.165112
188.462569,210.394983,45,0.1125,228,0.57,200.672194
210.394983,232.327398,50,0.125,278,0.695,222.573869
232.327398,254.259813,46,0.115,324,0.81,242.499317
254.259813,276.192228,37,0.0925,361,0.9025,264.404748
276.192228,298.124643,39,0.0975,400,1.0,287.042471


In [125]:
plot_freq_bar(x7_intervals, "Х7")

In [126]:
plot_polygon(x7_intervals, "Х7")

In [127]:
get_origin_binned_comparison(x7, x7_intervals)

source,mean,variance,std,skewness,excess
str,f64,f64,f64,f64,f64
"""origin""",197.899886,2993.25384,54.710637,0.034084,-1.10766
"""binned""",197.899886,2951.177397,54.32474,0.035399,-1.144972


In [129]:
get_estimation_of_population(x7_intervals, x7_stats)

mean,std,std_unbiased
f64,f64,f64
197.899886,54.32474,54.392774


## Х8

In [130]:
x8 = df.get_column("Х8")

In [131]:
x8_stats = get_col_stats(x8)
x8_stats

{'n': 400,
 'min': 124.55727570574376,
 'max': 250.8774512736014,
 'r': 126.32017556785763,
 'k': 9,
 'h': 14.035575063095292}

In [132]:
x8_bins = get_bins(x8_stats)
x8_bins

array([124.55727571, 138.59285077, 152.62842583, 166.6640009 ,
       180.69957596, 194.73515102, 208.77072608, 222.80630115,
       236.84187621, 250.87745127])

In [133]:
x8_intervals = get_interval_table(x8)
x8_intervals

interval_left,interval_right,freq,w,cum_freq,cum_w,mean
f64,f64,u32,f64,u32,f64,f64
124.557276,138.592851,50,0.125,50,0.125,130.689795
138.592851,152.628426,52,0.13,102,0.255,145.598301
152.628426,166.664001,43,0.1075,145,0.3625,160.029419
166.664001,180.699576,47,0.1175,192,0.48,172.440686
180.699576,194.735151,34,0.085,226,0.565,186.987893
194.735151,208.770726,44,0.11,270,0.675,201.41709
208.770726,222.806301,44,0.11,314,0.785,216.465004
222.806301,236.841876,44,0.11,358,0.895,230.438866
236.841876,250.877451,42,0.105,400,1.0,244.434119


In [134]:
plot_freq_bar(x8_intervals, "Х8")

In [135]:
plot_polygon(x8_intervals, "Х8")

In [136]:
get_origin_binned_comparison(x8, x8_intervals)

source,mean,variance,std,skewness,excess
str,f64,f64,f64,f64,f64
"""origin""",185.603806,1407.967017,37.522887,0.072784,-1.258332
"""binned""",185.603806,1392.630698,37.317967,0.074515,-1.283978


In [137]:
get_estimation_of_population(x8_intervals, x8_stats)

mean,std,std_unbiased
f64,f64,f64
185.603806,37.317967,37.364703


## Х9

In [138]:
x9 = df.get_column("Х9")

In [139]:
x9_stats = get_col_stats(x9)
x9_stats

{'n': 400,
 'min': 262.4500229206397,
 'max': 316.56364491738873,
 'r': 54.11362199674903,
 'k': 9,
 'h': 6.012624666305448}

In [140]:
x9_bins = get_bins(x9_stats)
x9_bins

array([262.45002292, 268.46264759, 274.47527225, 280.48789692,
       286.50052159, 292.51314625, 298.52577092, 304.53839558,
       310.55102025, 316.56364492])

In [141]:
x9_intervals = get_interval_table(x9)
x9_intervals

interval_left,interval_right,freq,w,cum_freq,cum_w,mean
f64,f64,u32,f64,u32,f64,f64
262.450023,268.462648,24,0.06,24,0.06,266.255777
268.462648,274.475272,48,0.12,72,0.18,271.738679
274.475272,280.487897,46,0.115,118,0.295,277.463193
280.487897,286.500522,61,0.1525,179,0.4475,283.381033
286.500522,292.513146,43,0.1075,222,0.555,289.230132
292.513146,298.525771,62,0.155,284,0.71,295.576806
298.525771,304.538396,46,0.115,330,0.825,301.419572
304.538396,310.55102,42,0.105,372,0.93,307.187716
310.55102,316.563645,28,0.07,400,1.0,312.964428


In [142]:
plot_freq_bar(x9_intervals, "Х9")

In [143]:
plot_polygon(x9_intervals, "Х9")

In [144]:
get_origin_binned_comparison(x9, x9_intervals)

source,mean,variance,std,skewness,excess
str,f64,f64,f64,f64,f64
"""origin""",289.439978,189.026049,13.748674,0.030007,-1.054181
"""binned""",289.439978,186.102112,13.641925,0.03428,-1.090562


In [145]:
get_estimation_of_population(x9_intervals, x9_stats)

mean,std,std_unbiased
f64,f64,f64
289.439978,13.641925,13.659009
